# Quality-filtering dashboard

Per-**device** quality gating of raw EIS/FSCV signal measurements: what was measured, and
which `(channel, timepoint)` sensors pass the checks that feed `discovery`. Run
**`raw_spectra_review.ipynb` first** to inspect the raw spectra and choose the stringency
parameters (especially `BAND`), then set them here.

Views are per device (default: every device at `ROOT`, each with its own plots).

---

**Where this fits — the ElectroPyCal analysis pipeline** (`pip install electropycal`):

1. `raw_spectra_review` — inspect raw EIS/FSCV spectra; choose the analysis band & QC stringency
2. ➡️ **`quality_filtering_dashboard` — set QC gates, see which `(channel, timepoint)` sensors pass, save the shared config  ← you are here**
3. `stabilization_review` — confirm each channel's FSCV response stabilized (optional / parallel)
4. `diagnostics_review` — characterize the featureset before modeling — *is recalibration well-posed?*
5. `discovery_checkpointed` — run the model-discovery task queue batch-by-batch under nested CV
6. `discovery_results_review` — inspect a finished discovery run; pick a model
7. `deployment_domain_shift` — freeze the chosen model; recalibrate in-vivo data; monitor CORAL domain shift

Consistency knobs (band, QC gates) are shared across notebooks via `<ROOT>/electropycal_analysis_config.json`. See `docs/USAGE.md`.


## Setup — run first

Installs `electropycal` from PyPI. On Colab it also mounts Google Drive, so you can point
`ROOT` at raw data in your own Drive; skip the mount if your data is local.

```
pip install electropycal
```

After installing on Colab, do **Runtime → Restart session** so the new module loads.


In [ ]:
import sys

if "google.colab" in sys.modules:
    # --upgrade is load-bearing. `pip install electropycal` is a NO-OP when any version is
    # already present, and a hosted runtime often carries an older one from an earlier
    # session, so without it the cell reports success and leaves the old version in place.
    !pip install -q --upgrade electropycal
    from google.colab import drive
    drive.mount("/content/drive")     # only needed if ROOT points into your Drive

# Check the version here, where the message is readable, instead of letting an older build
# fail deep inside a reader. If this raises immediately after an upgrade, restart the
# runtime: a module already imported stays in memory and the upgrade does not reach it.
import electropycal

_parts = electropycal.__version__.split(".")[:2]
if tuple(int(p) for p in _parts if p.isdigit()) < (0, 10):
    raise RuntimeError(
        f"electropycal {electropycal.__version__} is older than these notebooks need "
        f"(0.10 or newer, for the full range of export variants). Run "
        f"`pip install -q --upgrade electropycal`, then restart the runtime so the "
        f"upgraded version is the one imported.")

In [ ]:
%matplotlib inline
import os
for _v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
import tempfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from electropycal.viz import set_pub_style
from electropycal.qcdash import QCDashboard
set_pub_style()

# --- point ROOT at your data folder (Google Drive) ---
ROOT = None   # <- your own PSTrace export directory; None uses the shipped demo
# demo_input() locates demo/ from ANY working directory -- the repo root, notebooks/,
# or anywhere else -- and synthesizes an equivalent tree when no checkout is reachable
# (e.g. Colab). A relative literal here breaks in whichever cwd it was not written for.
# ROOT must be a LOCAL FILESYSTEM PATH. A Drive or cloud share URL will not work:
# nothing here downloads for you. On Colab, mount the drive first and use the
# mounted path:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     ROOT = '/content/drive/MyDrive/<your-folder>'
# Point ROOT at the directory that DIRECTLY CONTAINS the <YYYYMMDD>_..._signal
# folders, not at its parent. If you are one level off, extraction now says so and
# names the directory it thinks you meant.
from electropycal._demo import demo_input
ROOT = demo_input("in_vitro", ROOT)
print("ROOT =", ROOT)

## Available data — signal measurements by device / date / timepoint
`timepoint` = elapsed days since each device's first **signal** session (`D0`, `D1`, …). The
**channels** column is read from the header of each session's `<deviceid>_fscv_0nM.csv` file.

> This scan is deliberately **fast** — it only reads the *header line* of the 0 nM files to list
> channels, not the full waveforms or the EIS/dose files. The heavy parsing happens once, in the
> quality-check cell below. Only **signal** folders are read.

In [ ]:
# Selection is local to this run; QC/band parameters come from the shared config
# (<ROOT>/electropycal_analysis_config.json, set in raw_spectra_review). Override by rebuilding, e.g.
# QCDashboard(ROOT, device_types=["neurostring"], timepoints=[0,4,6], n_jobs=4) - see settings print.
DEVICE_TYPES = None      # None = all; or ["neurostring"], ["cfme"]
DEVICES      = None      # None = all ids; or ["2-2", "3-2"]
TIMEPOINTS   = None      # None = all; or day ints e.g. [0, 4, 6]
N_JOBS       = 4         # worker processes for parallel QC (-1 = all cores; identical result)

QC = QCDashboard(ROOT, devices=DEVICES, device_types=DEVICE_TYPES, timepoints=TIMEPOINTS, n_jobs=N_JOBS)
QC.print_settings()
print()
with pd.option_context("display.max_rows", None):
    display(QC.label_devices(QC.available_table()))

## Selection & quality-stringency parameters
Pick what to analyze by **device type / id / timepoint** (each defaults to *all available*), tune the
stringency knobs, and choose **which checks gate**.

**How each QC works** (all computed on the **replicate-averaged** measurement — EIS spectra averaged,
each concentration's first `REPLICATES` FSCV cycles averaged, *then* checked):

| code | check | how it's calculated | fails when |
|------|-------|---------------------|-----------|
| **E.1** | EIS.1 capacitive | sign of `Im(Z)` at **every** in-band frequency | `Im(Z) ≥ 0` anywhere in-band (non-capacitive / shunting) |
| **E.2** | EIS.2 \|Z\|-monotone | is `\|Z\|` non-increasing with `f` within `MONO_TOL` | `\|Z\|` rises with `f` beyond `MONO_TOL` (DC drift) |
| **E.3** | EIS.3 environment | sign of `Z'` across the band | `Z' < 0` somewhere (amplifier saturation) |
| **F.1…F.n** | FSCV reproducibility SNR **per concentration** | `NormIpeak / std(NormIpeak across cycles)` at that one dose vs `MIN_NORM_SNR` | `repeatability_snr < MIN_NORM_SNR` at that dose |
| **F.(n+1)** | FSCV dose-monotonic | Pearson `r` of `NormIpeak` vs `log10(concentration)` (≥3 doses) | `r < MONOTONIC_R_MIN` (or <3 concentrations) |
| **F.(n+2)** | FSCV `V_peak` in-window | fraction of doses whose `V_peak` is inside `PEAK_WINDOW` (not within `PEAK_EDGE_TOL` of a bound) | edge-pinned for **≥ half** the doses ("most concentrations" = a majority) |

`n` = number of measured concentrations (so with 5 doses: F.1–F.5 SNR, **F.6** monotonicity, **F.7** V_peak).

**Default recommendation (electropycal default).** Gate on **E.1–E.3 & dose-monotonicity** only — the
gate `extract_dataset(acceptance="monotonic")` uses. The per-concentration SNR (F.1…F.n) and V_peak
(F.n+2) are **reported but non-gating** by default: single-cycle SNR is hostage to imperfect
background matching, and edge-pinning is a window-fit flag, so neither is a hard reject. Add them to
`GATE_ON` below only if you want a stricter dataset.

> **`NormIpeak < 0` dose rows** (signal below background at `V_ox` — non-physical) are **dropped**
> before QC/discovery (`drop_negative=True`, reported per-dose as the `negative` column). They don't
> strand other doses: `NormIpeak` is normalized to the 0 nM background (not to another dose), and the
> feature D0-normalization anchors per sensor to the mean feature vector at its earliest timepoint.

**This is the heavy cell** — `channel_quality_report` does the full parse + QC (`progress=True`
prints per-session ETA).

In [ ]:
# (QC already ran when QCDashboard was constructed above — CH_TBL / DOSE_TBL and the check
# registry live on `QC`. Access them as QC.CH_TBL, QC.DOSE_TBL if you need the raw tables.)

## Quality checks — registry (per `GATE_ON` above)
The full check list with its **code**, **check** name, whether it **gates** `overall_valid` (from
your `GATE_ON`), how it's **calculated**, and its **fail condition**. `overall_valid` = a *paired*
channel passing every gating check. Change `GATE_ON` and re-run the QC cell to move a check in/out of
the gate. (Extraction also drops any single dose with `NormIpeak < 0` before the monotonic fit.)

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(QC.qc_definitions())

## 1. Measurement schedule
Uniform `16 × day` grid: every cell is the same size and timepoints sit at their real day
positions (only measured days are tick-labeled). Cells are colored by channel where the
sensor was measured; a red ✕ marks a `(channel, timepoint)` that fails quality.

In [ ]:
QC.plot_schedule()

## 2. Quality dropout by check
**Left:** channels *measured* vs channels *passing all gating checks* over time.
**Right:** one gapped bar per check at each timepoint — **colored** by check (E.1–E.3 a blues
gradient, F.1…F.n+2 an oranges gradient) for the channels that **pass**, **grey** for those that
**fail**. The legend maps each color to its check, so the x-axis is simply the timepoint.

In [ ]:
QC.plot_dropout_by_check()

## 3. Channel-timepoint quality table
One row per measured `(device, channel, timepoint)`: every check (E.1–E.3, F.1, F.2+), the
gating `overall_valid`, and the reasons any gating check failed.

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(QC.label_devices(QC.quality_table()))

## 4. Valid-sensor summary (per device-timepoint)
One row per `(device, date, timepoint)`: the valid and invalid channel counts + lists and the valid
fraction. This is the roster that carries forward to discovery.

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(QC.label_devices(QC.valid_sensor_summary()))

## 5. Cumulative QC-gate impact
How much raw data the gates remove overall, and per gate. **"Failed"** = fails **any** active gate
(`GATING_COLS`); **"retained"** = passes them all (`overall_valid`) — the set discovery trains on.
Per-gate failure counts **overlap** (one channel-timepoint can fail several gates), so they do **not**
sum to the total failed; the totals line reconciles paired = failed + retained.

In [ ]:
imp = QC.gate_impact()
with pd.option_context("display.max_rows", None):
    display(imp)

### 5.1 Dose-monotonicity stringency (FSCV.1)
**How it's computed:** each channel-timepoint's `dose_response_r` = correlation of `NormIpeak` with
`log10[DA]` across its non-zero doses — **Pearson** (linear-in-log, default) or **Spearman**
(rank/monotonic). Passes iff `dose_response_r >= MONOTONIC_R_MIN`. **Sweep:** channel-timepoints kept
vs the cutoff (current = red dashed). **r\*** (grey dashed) = the r at which one curve's correlation is
individually significant at p<0.05 for the typical dose count — a **reference, not the gate**.
**Caveats:** r\* is Pearson-derived and per-curve; QC is broad cleanup, so a cutoff **below** r\* is
expected (plateau vs cliff). Per-curve dose count varies (drop_negative / missing doses); set `RSTAR_N`
to fix the n used for r\*.

In [ ]:
QC.plot_dose_monotonicity_sweep()   # caches r / r* / Pearson-vs-Spearman for the stats artifact

> **Decision to record — FSCV.1 monotonicity gate (`mono_method`).** The sweep above shows, on the
> *real* sensors, how many `(channel, timepoint)` the **Pearson** vs **Spearman** dose-response gate
> accepts at each `monotonic_r_min`. **Look for:** does Spearman *rescue* genuinely saturating-but-
> monotone sensors that Pearson wrongly drops, without *admitting* noisy non-monotone ones? **Rule of
> thumb:** keep the **Pearson** default unless Spearman nets a clear gain in real, well-behaved
> sensors. **Record** the choice + the `monotonic_r_min` cutoff in the shared config (`mono_method`,
> `monotonic_r_min`) and re-save it — this is decision **3** in `docs/DESIGN.md`.

### 5.2 |Z|-monotonicity stringency (EIS.2)
**How it's computed:** `eis_z_rise` = the **max fractional |Z| rise** between adjacent in-band
frequencies (`max Δ|Z| / |Z|`, ascending in f). A healthy capacitive electrode has |Z| ~ 1/f (falls
with f) so its max rise is <= 0; a positive rise signals DC drift / disruption. Passes iff
`eis_z_rise <= MONO_TOL` (red dashed). **Caveats:** it's a *tolerance* (a small rise is allowed so
noise doesn't fail a good electrode), fractional (scale-free), and takes the single worst step — one
up-tick above MONO_TOL fails the spectrum.

In [ ]:
QC.plot_z_monotonicity_sweep()

### 5.3 Reproducibility-SNR stringency (per dose)
**How it's computed:** `repeatability_snr` per dose = the dose's `NormIpeak` divided by its
replicate-to-replicate noise (across the first `REPLICATES` FSCV cycles) — how far the signal sits
above measurement noise. The SNR gate (when `gate_on['snr_all']`) drops doses with
`repeatability_snr < MIN_NORM_SNR` (red dashed). **Caveats:** per **dose**, not per channel-timepoint;
**off by default** (`snr_all=False`); a genuine low-dose point can have low SNR without the electrode
being bad.

In [ ]:
QC.plot_snr_sweep()

## 6. Connection to the discovery dataset
The `overall_valid` roster above is what discovery trains/tests on: `discovery_checkpointed` calls
`extract_dataset(acceptance="monotonic")` with the **same knobs** (`BAND`, `MONO_TOL`,
`PEAK_METHOD`, `REPLICATES`, `MIN_NORM_SNR`, `MONOTONIC_R_MIN`) to build the featureset, then
`RunData.from_frame` consumes it.

Two things to know when reconciling this dashboard's valid set against `extract_dataset`'s kept set:

- **Gating must match.** With the default `GATE_ON` (E.1–E.3 + dose-monotonicity) the two coincide,
  because that *is* `acceptance="monotonic"`. If you add SNR / V_peak gates here, this dashboard is
  intentionally **stricter** than the default extraction, so the sets differ by design.
- **EIS frequency grid.** `extract_dataset` additionally drops a channel whose in-band EIS grid
  doesn't match the run's reference grid (its per-frequency features must stack onto one common
  axis). The per-channel QC here never compares channels, so a channel can pass every check yet be
  dropped by extraction on that grid constraint alone. (Set a `BAND` whose endpoints are shared by
  every session — see `raw_spectra_review` — to avoid it.)

### 6.1 Finalized-dataset overview — what discovery will train on
A **preview of the post-QC featureset** built from the QC tables above (no extraction/discovery run
needed). Two plots, one per cell:
- **Grid** — every sensor (`device:channel`) × timepoint, colored by how many concentrations survived
  QC for that channel-timepoint: **green = all doses retained, red = 0** (blank = no valid data). The
  valid-cell count matches the §5 'retained' total. Under the **default gates** (`acceptance='monotonic'`)
  a *valid* channel keeps **all** its doses, so a cell below the max means a dose was dropped by
  `drop_negative` (NormIpeak < 0) or was missing — not a per-dose SNR cut (that only applies under
  `snr`/`monotonic+snr`).
- **Traces (per sensor)** — `NormIpeak` vs timepoint in **true day spacing**, one line per sensor, a
  panel per concentration (y = mean NormIpeak per sensor-timepoint-dose). Each panel's y-top is
  clipped at the `YCLIP_PCT` percentile so a few outliers don't compress the rest (set `YMAX` for a
  hard cap). **Per-device average** — the same view, one trace per device (mean over its sensors) with
  a ±1 s.d. band, to see device-level drift.

In [ ]:
QC.plot_finalized_overview()   # §6.1 grid + per-sensor + per-device (reuses electropycal.overview)

In [ ]:
# (per-sensor NormIpeak-vs-time is included in QC.plot_finalized_overview() above)

In [ ]:
# (per-device average is included in QC.plot_finalized_overview() above)

## 7. QC statistics artifact — save what the gates did

Companion to the shared **config** (which records *what* QC parameters were used), this writes
`<ROOT>/electropycal_qc_stats.json` recording *what they did* on this data: overall yield, per-gate
dropout, the dose-monotonicity / SNR / EIS.2 stringency numbers, and the Pearson-vs-Spearman
comparison. Together, config + stats let you (and a reviewer) see both the knobs and their effect on
the finalized dataset. Re-run after any parameter change to refresh it.

In [ ]:
QC.save_stats()   # writes <ROOT>/electropycal_qc_stats.json from QC's own tables (no globals)